# Summarize One Scenario Run

This notebook produces a compact end-year summary table (`summary.csv`) and a quick statistics table for one run folder.


## What Is Done

1. Load every `*/output.csv` in one run folder.
2. Add annualized investment/subsidy rows at end-year.
3. Save end-year metrics by scenario to `summary.csv` and print describe/reference/extrema stats.


In [ ]:
from pathlib import Path

from project.analysis.post_processing.shared.io_runs import concat_output_bundle, load_output_bundle
from project.analysis.post_processing.shared.transforms import (
    add_annualized_rows,
    describe_with_reference_and_extrema,
    extract_end_year_values,
)


In [ ]:
run_id = "policies_scenarios_market_failures"
reference = "Reference"


def resolve_run_folder(run_name: str) -> Path:
    candidate_roots = [
        Path("project/analysis/post_processing/policy_decomposition/runs/complete_interactions"),
        Path("project/analysis/post_processing/policy_decomposition/runs/many_scenarios"),
        Path("runs/complete_interactions"),
        Path("runs/many_scenarios"),
    ]

    for root in candidate_roots:
        candidate = (root / run_name).resolve()
        if candidate.exists() and any(candidate.glob("*/output.csv")):
            return candidate

    searched = "\n".join(str((root / run_name).resolve()) for root in candidate_roots)
    raise FileNotFoundError(f"Run folder not found for '{run_name}'. Checked:\n{searched}")


folder = resolve_run_folder(run_id)
print(f"Run folder: {folder}")


In [ ]:
variables = [
    "Consumption (TWh)",
    "Emission (MtCO2)",
    "Energy poverty (Million)",
    "Investment (Billion euro/year)",
    "Subsidies (Billion euro/year)",
    "Investment insulation (Billion euro/year)",
    "Investment heater (Billion euro/year)",
]

annualized_mapping = {
    "Investment heater (Billion euro/year)": "Investment heater (Billion euro)",
    "Investment insulation (Billion euro/year)": "Investment insulation (Billion euro)",
    "Investment (Billion euro/year)": "Investment total (Billion euro)",
    "Subsidies (Billion euro/year)": "Subsidies total (Billion euro)",
}

ignore = ["root_log.log", "img", ".DS_Store", "comparison.csv", "consumption.png", "summary.csv"]
summary_path = folder / "summary.csv"


In [ ]:
bundle = load_output_bundle(folder, ignore=ignore)
if not bundle:
    raise FileNotFoundError(f"No scenario output files found in: {folder}")

bundle = {scenario: add_annualized_rows(df, annualized_mapping) for scenario, df in bundle.items()}
data = concat_output_bundle(bundle)

summary = extract_end_year_values(data, variables)
summary.sort_index().to_csv(summary_path)

result = describe_with_reference_and_extrema(summary, reference_name=reference, reference_label="reference")
result = result.loc[:, variables]

print(f"Scenarios loaded: {len(bundle)}")
print(f"Summary saved: {summary_path}")
result
